# 03 從雜亂的造冊到乾淨的資料 — 筆記本

Jun 從市衛生局匯出了颱風後的通報資料：`santa_luz_linelist_raw.csv`，637 列，由許多人經手打字。在這本筆記本裡，你要一小步、一小步地檢查，把它變成後面每一章都使用的、乾淨而且已去識別化的 line list（造冊）。

> **這裡的一切都是虛構的。** Santa Luz 市、颱風 Amihan，以及原始檔案裡的每個姓名、電話和住址，都是合成的訓練資料。姓名和號碼只是*看起來*像真的，好讓你練習如何保護它們。

計畫是：**載入並檢視 → 保護身分 → 修正日期 → 修正地點 → 修正代碼 → 修正年齡 → 移除重複 → 驗證 → 證明。**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

## 步驟 1 · 載入並檢視

檔案開頭有一行說明，所以我們略過一列。我們也要求 pandas 把**每一欄都當成文字**讀進來（`dtype=str`），並讓空白的儲存格維持空字串（`keep_default_na=False`）。為什麼？如果 pandas 在一份雜亂的檔案上去猜型別，它會悄悄地改動東西：`0900...` 電話號碼會失去開頭的零，`N/A` 會變成缺漏值，`46227` 會變成數字。我們想要看到 Jun 當初看到的那一團亂，然後自己決定每一項改動。

我們只看幾個不會識別個人的欄位。

In [ ]:
import hashlib
import hmac
import io
import secrets

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# BEGIN-CJK (中文字型設定：避免標籤顯示為方框)
import pathlib
import matplotlib.font_manager as fm
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass
_discovered = []
for _entry in fm.fontManager.ttflist:
    if any(_kw in _entry.name.lower() for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = ["Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP", "Noto Sans TC",
              "Microsoft JhengHei", "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
              "Heiti TC", "DejaVu Sans"]
_cjk_fonts = _discovered + [n for n in _preferred if n not in _discovered]
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts
# END-CJK

RAW_PATH = "data/synthetic/ph/santa_luz_linelist_raw.csv"

# skiprows=1 略過說明那一行；dtype=str 讓每個儲存格維持文字；
# keep_default_na=False 讓空白儲存格維持 "" 而不是 NaN
raw = pd.read_csv(RAW_PATH, skiprows=1, dtype=str, keep_default_na=False)

print(f"{raw.shape[0]} 列，{raw.shape[1]} 欄")
print(list(raw.columns))
raw[["Date Reported", "Sex", "Age", "Barangay", "Disease", "Date of Onset"]].head(8)

## 步驟 2 · 隱私優先：先保護身分，再清理

第 02 章教過規則：**識別資訊要在分析開始之前離開表格。** 我們仍然需要姓名來做一件事，就是找出被通報兩次的同一位病患，所以我們先把它轉成第 02 章的**加鹽假名 ID**：同一個整理過的姓名永遠得到同一個 ID，但沒有秘密的鹽，沒有人能重做它，也無法從中讀回姓名。然後我們刪掉姓名、電話、住址、通報單位和自由文字備註（備註也可能點名*別人*）。

在同一個儲存格裡，我們也把欄位名稱統一成 `lower_case_with_underscores`，這樣比較好打字。

> **問問 Tala：**「我有一個 pandas DataFrame `raw`，有文字欄位 `Patient Name`、`Contact No.`、`Address`、`Reporting Unit`、`Remarks` 和其他欄位。虛構的範例：`Patient Name = 'Juana X. Halimbawa'`。請用我的函式 `pseudo_id(name, salt)` 加上一個 `pseudo_id` 欄，刪掉那五個欄位，並把所有欄位名稱改成 snake_case。」

In [ ]:
SALT = secrets.token_hex(16)  # 僅為示範：實際工作時，只建立一次並鎖起來（第 02 章）


def pseudo_id(name: str, salt: str) -> str:
    key = " ".join(name.upper().replace(".", " ").split())   # 將姓名正規化
    digest = hmac.new(salt.encode(), key.encode(), hashlib.sha256).hexdigest()
    return "P-" + digest[:10]


df = raw.copy()
df.insert(0, "pseudo_id", df["Patient Name"].map(lambda n: pseudo_id(n, SALT)))
df = df.drop(columns=["Patient Name", "Contact No.", "Address", "Reporting Unit", "Remarks"])

# "Date of Onset" -> "date_of_onset"，"Did Cleanup Work" -> "did_cleanup_work"
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
print(list(df.columns))

# 檢查：同一個姓名用兩種方式打 -> 同一個 ID；不同姓名 -> 不同 ID；沒有識別資訊殘留
assert pseudo_id("Juana X. Halimbawa", SALT) == pseudo_id("JUANA X  HALIMBAWA", SALT)  # 虛構的姓名
assert pseudo_id("Juana X. Halimbawa", SALT) != pseudo_id("Juan X. Halimbawa", SALT)
for column in ["patient_name", "contact_no", "address", "reporting_unit", "remarks"]:
    assert column not in df.columns
print("識別資訊已移除；ID 長這樣：", df["pseudo_id"].iloc[0])

## 步驟 3 · Tala 搞錯了：無聲的日期對調

Jun 的日期有許多樣子：`07/24/2026`、`7/24/2026`、`2026-07-24`、`24-Jul-26`、`July 24, 2026`，還有像 `46227` 這樣的純數字。

> **問問 Tala：**「我的欄位 `date_of_onset` 裡的日期格式很混亂。請把它轉成 datetime。」

Tala 很快就回答：*「使用 `pd.to_datetime(..., format='mixed', dayfirst=True)`，因為世界上大多數地方都是日在前。」* 程式碼執行時沒有任何錯誤。這就是危險之處。我們把 Tala 的答案放在獨立的變數裡，這樣它不會弄壞我們的資料，並用護理師會問的兩個問題來測試它：**有幾個日期失敗了？** 以及 **有沒有人在就診之後才發病？**

In [ ]:
# Tala 的第一個答案：另外存放，這樣它不會弄壞 df
naive_onset = pd.to_datetime(df["date_of_onset"], format="mixed", dayfirst=True, errors="coerce")
naive_consult = pd.to_datetime(df["date_consulted"], format="mixed", dayfirst=True, errors="coerce")

print("變成 NaT（不是時間）的發病日：", naive_onset.isna().sum())
print("發病日晚於就診日的列數：", (naive_onset > naive_consult).sum())

swapped = naive_onset > naive_consult
pd.DataFrame({"發病日（原始輸入）": df.loc[swapped, "date_of_onset"],
              "就診日（原始輸入）": df.loc[swapped, "date_consulted"],
              "Tala 把就診日讀成": naive_consult[swapped].dt.date}).head(6)

兩個問題，而且都是無聲的：

1. **22 個發病日變成了 `NaT`**，也就是 Excel 序列數字（`46227`）。對 pandas 來說，它們根本不是日期。
2. **70 位病患現在「在就診之後才發病」。** 用了 `dayfirst=True`，`8/1/2026`（8 月 1 日）被讀成 1 月 8 日，連 `2026-08-01` 也一樣。像 `7/29/2026` 這樣的日期還是讀對了（沒有 29 月），所以錯誤藏在正確的資料列之間。這 70 筆當中，只有 2 筆是 Jun 檔案裡真正的問題；其他 68 筆是 Tala 造成的。

在菲律賓，斜線格式是月／日／年，而且 Jun 確認這份匯出檔裡每一個斜線日期都是這樣。所以我們**每種格式寫一條規則**，並拒絕猜測。

> **問問 Tala（第二次嘗試）：**「請寫一個函式 `parse_dates(s)`，處理一個由文字日期組成的 pandas Series。格式有：`M/D/YYYY`（月在前，前導零可有可無）、`YYYY-MM-DD`、`DD-Mon-YY`（年份是 20YY）、有或沒有逗號的 `Month D YYYY`，以及 5 位數的 Excel 序列數字（從 1899-12-30 起算的天數）。空白代表缺漏。請明確地逐一解析每種格式；不要讓 pandas 去猜。」

In [ ]:
def parse_dates(s: pd.Series) -> pd.Series:
    """逐一以明確的格式，解析 Jun 的各種日期樣式。"""
    s = s.str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    serial = s.str.fullmatch(r"\d{5}")  # Excel 從 1899 年 12 月 30 日起算天數
    out[serial] = pd.to_datetime(s[serial].astype(int), unit="D", origin="1899-12-30")
    for fmt in ["%m/%d/%Y", "%Y-%m-%d", "%d-%b-%y", "%B %d %Y"]:
        attempt = pd.to_datetime(s.str.replace(",", "", regex=False), format=fmt, errors="coerce")
        out = out.fillna(attempt)  # 只填補仍然缺漏的日期
    return out


DATE_COLUMNS = {"date_reported": "date_report", "date_of_onset": "date_onset",
                "date_consulted": "date_consult", "date_lab_confirmed": "date_lab_confirmed"}
for old, new in DATE_COLUMNS.items():
    text = df[old]
    parsed = parse_dates(text)
    not_understood = parsed.isna() & text.str.strip().ne("")
    assert not_understood.sum() == 0, text[not_understood].unique()
    df[new] = parsed
    print(f"{new:<20} 空白：{text.str.strip().eq('').sum():>3}   已解析：{parsed.notna().sum()}")

**檢查一下。** 先用一個可以用眼睛檢查的小案例：同一天的七種寫法。然後再問和剛才一樣的兩個問題。

In [ ]:
same_day = pd.Series(["46227", "7/24/2026", "07/24/2026", "2026-07-24",
                      "24-Jul-26", "July 24 2026", "July 24, 2026", ""])
result = parse_dates(same_day)
print(result.dt.date.tolist())
assert (result[:7] == pd.Timestamp("2026-07-24")).all()
assert pd.isna(result[7])

print("是 NaT 的發病日：", df["date_onset"].isna().sum())
print("發病日晚於就診日的列數：", (df["date_onset"] > df["date_consult"]).sum())

從 70 筆不可能的資料列，降到 **2** 筆。那兩筆是原始檔案裡真正的打字錯誤（發病日晚打了一個月），我們在步驟 7 處理。

## 步驟 4 · 一個 barangay，許多種拼法

`Ilog`、`Brgy. Ilog`、`ILOG`、`Bgy Ilog `（結尾有空白）、`Barangay Ilog`……再加上把 Santo Niño 寫成 `Sto. Nino`，還有幾個錯字。我們建立一個**鍵（key）**：移除重音符號、轉小寫、去掉 `barangay/brgy/bgy` 前綴、把 `sto` 寫成 `santo`、把連字號和句點換成空白，並修正三個已知的錯字。我們用官方的 barangay（描籠涯，菲律賓最基層的行政區，類似台灣的村里）名單做出同樣的鍵，再以它來合併，這也會帶進 `barangay_id` 和 `band`。

**檢查一下：** 只要有任何一列找不到 barangay，`assert` 就會讓筆記本停下來。也注意一下數量：各種拼法變成 23 個 barangay，因為第 24 個，山區的 Bukid，完全沒有通報。請在第 04 章記得這件事：病例數為零的 barangay 仍然屬於這張表。

In [ ]:
TYPOS = {"pobalcion": "poblacion", "dalampasig": "dalampasigan", "kawayan": "kawayanan"}


def barangay_key(s: pd.Series) -> pd.Series:
    """不怕拼法差異的鍵：'Brgy. Sto. Niño ' 和 'santo nino' 都會變成 'santo nino'。"""
    key = (s.str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")  # ñ -> n
           .str.lower().str.strip()
           .str.replace(r"^(barangay|brgy\.?|bgy\.?)\s+", "", regex=True)
           .str.replace(r"^sto\.?\s+", "santo ", regex=True)
           .str.replace(r"[-.]", " ", regex=True)
           .str.split().str.join(" "))
    return key.replace(TYPOS)


barangays = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")
barangays["key"] = barangay_key(barangays["barangay"])

df = df.rename(columns={"barangay": "barangay_raw"})
df["key"] = barangay_key(df["barangay_raw"])
df = df.merge(barangays[["key", "barangay_id", "barangay", "band"]], on="key", how="left")

unmatched = df.loc[df["barangay_id"].isna(), "barangay_raw"].unique()
assert len(unmatched) == 0, unmatched  # 檢查一下：每一列都找到了它的 barangay
print("原始拼法數：", df["barangay_raw"].nunique(), "-> 官方 barangay 數：", df["barangay"].nunique())

# 每個 barangay 有幾種拼法？Jun 可能想拿這張圖給登打人員看。
spellings = df.groupby("barangay")["barangay_raw"].nunique().sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(spellings.index, spellings.values, color="#0038A8")
ax.set_xlabel("Jun 匯出檔中不同的拼法數")
ax.set_title("一個 barangay，許多種拼法")
plt.tight_layout()
plt.show()

print(df.loc[df["barangay"] == "Santo Niño", "barangay_raw"].unique())

## 步驟 5 · Taglish 的代碼：性別、疾病、分類、是／否、結局、嚴重程度

每個欄位都有一個小小的**對照表**（字典）或一條規則。不在表裡的東西會變成缺漏，所以儲存格結尾的檢查會抓到任何我們忘記的拼法。

| 欄位 | 原始範例 | 乾淨的結果 |
|---|---|---|
| Sex | `M`、`male`、`lalaki` / `F`、`Female`、`babae` | `M` / `F` |
| Disease | `Lepto`、`LEPTO`…… / `DENGUE`、`Dengue Fever`、`DF` | `leptospirosis` / `dengue` |
| Classification | `lab confirmed`、`Conf` / `prob.` / `sus`、`Suspected` | `confirmed` / `probable` / `suspect` |
| Yes/no | `Y`、`yes`、`oo` / `N`、`no`、`hindi` / 空白、`N/A` | `1` / `0` / 缺漏 |
| Outcome | `buhay`、`A` / `Namatay`、`Expired`、`D` | `alive` / `died` |

In [ ]:
sex = df["sex"].str.strip().str.lower()
df["sex"] = sex.map({"m": "M", "male": "M", "lalaki": "M", "f": "F", "female": "F", "babae": "F"})

disease = df["disease"].str.strip().str.lower()
df["disease"] = np.select([disease.str.startswith("lep"),
                           disease.str.startswith("deng") | disease.eq("df")],
                          ["leptospirosis", "dengue"], default=None)

cls = df["classification"].str.strip().str.lower()
df["case_class"] = np.select([cls.str.contains("conf"), cls.str.startswith("prob"),
                              cls.str.startswith("sus")],
                             ["confirmed", "probable", "suspect"], default=None)

YES_NO = {"y": 1, "yes": 1, "oo": 1, "n": 0, "no": 0, "hindi": 0}  # 空白和 "n/a" -> 缺漏
for old, new in {"admitted": "hospitalised", "waded_in_floodwater": "waded_floodwater",
                 "did_cleanup_work": "cleanup_work"}.items():
    df[new] = df[old].str.strip().str.lower().map(YES_NO).astype("Int64")

outcome = df["outcome"].str.strip().str.lower()
df["outcome"] = outcome.map({"alive": "alive", "a": "alive", "buhay": "alive",
                             "died": "died", "d": "died", "namatay": "died", "expired": "died"})

sev = df["dengue_severity"].str.strip().str.lower()
df["dengue_severity"] = np.select(  # 順序很重要："without" 也包含 "with"
    [sev.str.contains("severe") | sev.eq("sd"),
     sev.str.contains("without") | sev.str.contains("w/o", regex=False) | sev.eq("no ws"),
     sev.str.contains("with") | sev.str.contains("w/", regex=False) | sev.eq("ws+")],
    ["severe", "no_warning", "warning_signs"], default=None)
df.loc[df["disease"] != "dengue", "dengue_severity"] = None

# 檢查：沒有東西從對照表中漏掉
for column in ["sex", "disease", "case_class", "outcome", "hospitalised"]:
    assert df[column].notna().all(), column
assert df.loc[df["disease"] == "dengue", "dengue_severity"].notna().all()
print(df["disease"].value_counts().to_dict(), df["sex"].value_counts().to_dict())

## 步驟 6 · 年齡：「34 y/o」、「8 mos」和一個 150

取第一個整數。如果文字裡含有 `mo`，那就是月數，所以除以 12 並保留整年數（`8 mos` → 0，就像病例調查表上對嬰兒的做法）。超過 110 的年齡是不可能的，所以變成缺漏；現實中你會打電話給通報單位。然後我們把年齡放進標準的年齡組。

In [ ]:
def parse_age(s: pd.Series) -> pd.Series:
    """從像 '34'、'34 y/o'、'34yo'、'8 mos'、'8 months' 的文字取出完整年數。"""
    text = s.str.strip().str.lower()
    number = text.str.extract(r"^(\d+)")[0].astype("Int64")
    years = number.where(~text.str.contains("mo"), number // 12)  # 月 -> 年
    return years.where(years <= 110).astype("Int64")             # 150 -> 缺漏


# 用一個能在腦中算出來的案例檢查
print(parse_age(pd.Series(["34", "34 y/o", "34yo", "8 mos", "18 months", "150", ""])).tolist())

df["age"] = parse_age(df["age"])
df["age_group"] = pd.cut(df["age"], bins=[-1, 4, 14, 24, 44, 64, 200],
                         labels=["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]).astype(object)
print("年齡缺漏：", df["age"].isna().sum(), "| 最年長：", df["age"].max())

## 步驟 7 · 重複資料、不可能的日期，以及新的 ID

**完全相同的副本**很容易處理：`df.duplicated()`。**近似重複**比較難：市衛生局曾經重新通報過一些病患，姓名重新打字、日期格式不同、barangay 拼法也不一樣。經過我們的清理之後，這些列在重要的欄位上*看起來*一樣，所以我們依據（**假名 ID、疾病、發病日**）去重，並保留第一筆。

然後是最後三條規則：

- **不可能的日期：** 如果發病日晚於就診日（或就診日空白時，晚於通報日），就把發病日設為缺漏；現實中，要打電話給通報單位。
- **假名病例 ID** `SLC-0001`、`SLC-0002`、……，依檔案順序編號。
- **欄位**依約定的順序排列。假名 ID、原始文字和識別資訊都留在後面。

In [ ]:
print("完全相同的副本：", df.duplicated().sum())
duplicate = df.duplicated(subset=["pseudo_id", "disease", "date_onset"], keep="first")
print("同一人 + 疾病 + 發病日（完全相同與近似重複）：", duplicate.sum())
df = df[~duplicate].reset_index(drop=True)

later = df["date_consult"].fillna(df["date_report"])
impossible = df["date_onset"] > later
print("發病日晚於就診日／通報日 -> 設為缺漏：", impossible.sum())
df.loc[impossible, "date_onset"] = pd.NaT

df["case_id"] = [f"SLC-{i:04d}" for i in range(1, len(df) + 1)]

COLUMNS = ["case_id", "disease", "case_class", "date_onset", "date_consult", "date_report",
           "date_lab_confirmed", "age", "age_group", "sex", "barangay_id", "barangay", "band",
           "hospitalised", "outcome", "dengue_severity", "waded_floodwater", "cleanup_work"]
clean = df[COLUMNS]
print(clean.shape)
clean.head()

## 步驟 8 · 把驗證規則寫成斷言

**驗證規則**是關於資料、永遠必須為真的一句話，寫成 `assert`。如果未來的匯出檔違反了某條規則，筆記本會大聲地停下來，而不是悄悄地產出一份錯誤的報告。把每一行唸出來：它們是流行病學，不只是程式碼。

In [ ]:
assert clean["case_id"].is_unique
assert clean["date_report"].between(pd.Timestamp("2026-07-20"), pd.Timestamp("2026-09-30")).all()
both = clean.dropna(subset=["date_onset", "date_consult"])
assert (both["date_onset"] <= both["date_consult"]).all()          # 就診之前就已經生病
assert (clean["date_consult"] <= clean["date_report"]).where(clean["date_consult"].notna(), True).all()
assert clean["age"].dropna().between(0, 110).all()
assert set(clean["sex"]) == {"M", "F"}
confirmed = clean["case_class"] == "confirmed"
assert clean.loc[confirmed, "date_lab_confirmed"].notna().all()     # 確診 = 有實驗室結果
assert clean.loc[clean["disease"] == "dengue", "waded_floodwater"].isna().all()
print("所有驗證規則都通過。")
print(clean["disease"].value_counts().to_dict())
print("缺漏：發病日", clean["date_onset"].isna().sum(), "| 就診日", clean["date_consult"].isna().sum(),
      "| 年齡", clean["age"].isna().sum())

## 步驟 9 · 證明它

Doc Mon 的問題：*「你的和官方的乾淨檔案一樣嗎？」* 我們依照 C8 規則的方式儲存我們的表格（`to_csv(index=False)`，這裡存在記憶體中），再讀回來，讓 pandas 逐個儲存格和 `santa_luz_linelist.csv` 比對。當兩張表完全相同時，`assert_frame_equal` 什麼都不會印出；不相同時，它會說明第一個差異。

In [ ]:
csv_text = clean.to_csv(index=False, lineterminator="\n")  # 我們會儲存的內容
ours = pd.read_csv(io.StringIO(csv_text))
official = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

pd.testing.assert_frame_equal(ours, official)
print(f"與 santa_luz_linelist.csv 完全相同：{ours.shape[0]} 列 x {ours.shape[1]} 欄")

## 換你試試

1. **菲律賓語的月份名稱。** 一位新的登打人員打了 `Hulyo 24, 2026`。在新的儲存格裡執行 `parse_dates(pd.Series(["Hulyo 24, 2026"]))`。你得到什麼？步驟 3 的 `assert` 會保護你嗎？*提示：在解析之前，先用 `.str.replace` 把 `Hulyo` 換成 `July`（以及把 `Agosto` 換成 `August`），再重新執行七種寫法的檢查。*
2. **一種新的拼法。** `barangay_key(pd.Series(["Brgy Tabing  Ilog"]))` 會回傳什麼？它和 `tabing ilog` 相符嗎？現在試試 `"Tabing-ilog (riverside)"`。*提示：鍵不可能修正所有東西。用 `.str.contains("Tabing")` 找出那一列，只有在你知道登打人員的意思時，才加一條規則。*
3. **算算工作量。** 每一個步驟改動了多少列？*提示：比較 `len(raw)` 和 `len(clean)`，並比較 `naive_onset.isna()` 與 `df["date_onset"].isna()` 的計數。把這些數字寫成給 Jun 的三行說明。*